# AskJack — Préparation des données & vectorisation (incrémental)

Pipeline : **sources → txt → chunks + meta → index FAISS**, piloté par `config.yaml` + `datasource_conf.yaml` (sources, regex projet).

Convertisseurs : PDF → `pymupdf4llm` (Markdown) · docx → `MarkItDown` (Markdown) · xlsx/xls → une ligne par ligne de tableau · autres → texte brut.

| Étape | Périmètre |
|---|---|
| 1. Inventaire | sources (globs de la config) comparées aux `.txt` existants |
| 2. Extraction | **uniquement** les sources nouvelles ou plus récentes que leur `.txt` |
| 3. Manifeste + chunking | **tous** les `.txt` (retouchés à la main compris) |
| 4. Vectorisation | reconstruction complète de l'index à partir de `chunks.jsonl` |

Les `.txt` reproduisent l'arborescence des sources : `data/fiches-projets/P5_x.docx` → `<txt_dir>/data/fiches-projets/P5_x.docx.txt`.
Le `.txt` suffit donc à retrouver sa source, son `doc_type` et son projet.
Les anciens `.txt` « à plat » sont migrés automatiquement à l'étape 1 (date conservée, retouches préservées).

⚠️ Une source modifiée **après** la retouche manuelle de son `.txt` sera ré-extraite (la retouche est écrasée) : l'étape 1 les signale.

## 0. Setup

In [ ]:
# %pip install pyyaml pymupdf4llm "markitdown[docx]" openpyxl python-dotenv pandas langchain-community langchain-mistralai langchain-text-splitters faiss-cpu
import json, os, re, sys
from datetime import datetime
from fnmatch import fnmatch
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")
assert os.getenv("MISTRAL_API_KEY"), "MISTRAL_API_KEY manquante (.env)"

from src import rag
CFG = rag.load_config(ROOT)          # config.yaml + datasource_conf.yaml fusionnés
print(f"{len(CFG['sources'])} sources déclarées | regex projet : {CFG['project_id_regex']}")

TXT_DIR     = ROOT / CFG["paths"]["txt_dir"]
CHUNKS_FILE = ROOT / CFG["paths"]["chunks_file"]
INDEX_DIR   = ROOT / CFG["paths"]["index_dir"]
for p in (TXT_DIR, CHUNKS_FILE.parent, INDEX_DIR):
    p.mkdir(parents=True, exist_ok=True)

FORCE_ALL = False   # True = ré-extraire toutes les sources (écrase les retouches manuelles !)

def txt_path(source_path: str) -> Path:
    """data/x/P5_a.docx -> <TXT_DIR>/data/x/P5_a.docx.txt"""
    return TXT_DIR / (source_path + ".txt")

def source_of(txt: Path) -> str:
    """Inverse de txt_path : <TXT_DIR>/data/x/P5_a.docx.txt -> data/x/P5_a.docx"""
    return txt.relative_to(TXT_DIR).as_posix()[:-len(".txt")]

def extract_project_id(name: str):
    m = re.match(CFG["project_id_regex"], name)
    return int(m.group(1)) if m else None

def source_conf(source_path: str):
    """1re entrée de config.sources dont le glob correspond (même règle que l'inventaire)."""
    return next((s for s in CFG["sources"] if fnmatch(source_path, s["glob"])), None)

fmt_dt = lambda ts: datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M") if ts else "-"
print("ROOT :", ROOT)

## 1. Inventaire : sources à convertir

In [ ]:
rows, seen, migrated = [], set(), 0
for src in CFG["sources"]:
    for path in sorted(ROOT.glob(src["glob"])):
        sp = path.relative_to(ROOT).as_posix()
        if not path.is_file() or sp in seen:       # 1er glob qui matche gagne
            continue
        seen.add(sp)
        txt = txt_path(sp)
        legacy = TXT_DIR / (path.name + ".txt")          # ancien format "à plat" -> migré (date conservée)
        if legacy.exists() and legacy != txt and not txt.exists():
            txt.parent.mkdir(parents=True, exist_ok=True)
            os.replace(legacy, txt)
            migrated += 1
        src_mtime = path.stat().st_mtime
        txt_mtime = txt.stat().st_mtime if txt.exists() else None
        status = ("à convertir (nouveau)" if txt_mtime is None
                  else "à convertir (source modifiée)" if src_mtime > txt_mtime
                  else "à jour")
        if FORCE_ALL and status == "à jour":
            status = "à convertir (forcé)"
        rows.append({"source_path": sp, "doc_type": src["doc_type"], "status": status,
                     "source_modif": fmt_dt(src_mtime), "txt_modif": fmt_dt(txt_mtime)})

inventory = pd.DataFrame(rows)
todo = inventory[inventory.status.str.startswith("à convertir")]
existing_txt = sorted(TXT_DIR.rglob("*.txt"))
orphans = [t for t in existing_txt if source_of(t) not in seen]

if migrated: print(f"(migration : {migrated} .txt à plat déplacés vers l'arborescence miroir)")
print(f"Sources trouvées           : {len(inventory)}")
print(f"Fichiers texte existants   : {len(existing_txt)}")
print(f"  dont à jour              : {(inventory.status == 'à jour').sum()}")
print(f"  dont orphelins (source absente ou hors config) : {len(orphans)}")
print(f"À convertir                : {len(todo)}")
print(todo.status.value_counts().to_string() if len(todo) else "  -> rien à faire")
todo

In [ ]:
# Sources modifiées : leur .txt (peut-être retouché à la main) sera écrasé à l'étape 2
inventory[inventory.status == "à convertir (source modifiée)"]

In [ ]:
# Orphelins : .txt sans source (supprimée, renommée, ou glob retiré de la config).
# Ils sont quand même chunkés à l'étape 3 s'ils matchent encore un glob ; sinon ignorés.
[source_of(t) for t in orphans]

## 2. Extraction texte (uniquement les sources à convertir)

In [ ]:
import pymupdf4llm
from markitdown import MarkItDown

_md = MarkItDown()

def read_pdf(path: Path) -> str:
    return pymupdf4llm.to_markdown(str(path))

def read_docx(path: Path) -> str:
    return _md.convert(str(path)).text_content

def read_xlsx(path: Path) -> str:
    """Une ligne de texte par ligne de tableau ('Colonne : valeur | ...') -> chunkable normalement."""
    parts = []
    for name, df in pd.read_excel(path, sheet_name=None).items():
        df = df.dropna(how="all").dropna(axis=1, how="all")
        lines = [" | ".join(f"{c} : {v}" for c, v in row.items() if pd.notna(v))
                 for _, row in df.iterrows()]
        if lines:
            parts.append(f"## Feuille : {name}\n\n" + "\n\n".join(lines))
    return "\n\n".join(parts)

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

READERS = {".pdf": read_pdf, ".docx": read_docx, ".xlsx": read_xlsx, ".xls": read_xlsx}   # .xls : %pip install xlrd

def extract(path: Path) -> str:
    text = READERS.get(path.suffix.lower(), read_text)(path)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

log = []
for sp in todo.source_path:
    out = txt_path(sp)
    out.parent.mkdir(parents=True, exist_ok=True)
    try:
        text = extract(ROOT / sp)
        out.write_text(text, encoding="utf-8")
        log.append({"source_path": sp, "n_chars": len(text), "erreur": None})
    except Exception as e:
        log.append({"source_path": sp, "n_chars": 0, "erreur": repr(e)})

log = pd.DataFrame(log, columns=["source_path", "n_chars", "erreur"])
print(f"{(log.erreur.isna()).sum()} converti(s), {log.erreur.notna().sum()} en erreur")
log.sort_values("n_chars")

In [ ]:
# Contrôle : extractions vides ou suspectes (PDF image, etc.)
log[(log.n_chars < 200) | log.erreur.notna()]

## 3. Manifeste (reconstruit depuis les `.txt`) + chunking → `chunks_file`

Point de départ : **tous** les `.txt` présents. Pour chacun, la source est retrouvée par le chemin,
puis `doc_type` / `label` / `no_split` par les globs de la config et `project_id` par la regex.

In [ ]:
manifest, skipped = [], []
for txt in sorted(TXT_DIR.rglob("*.txt")):
    sp = source_of(txt)
    conf = source_conf(sp)
    if conf is None:
        skipped.append(sp)
        continue
    manifest.append({
        "source_path": sp,
        "doc_type":    conf["doc_type"],
        "label":       conf["label"],
        "project_id":  extract_project_id(Path(sp).name),
        "no_split":    conf.get("no_split", False),
        "n_chars":     len(txt.read_text(encoding="utf-8")),
    })
manifest = pd.DataFrame(manifest)

print(f"{len(manifest)} fichiers texte retenus, {len(skipped)} ignorés (aucun glob de la config)")
if skipped: print("  ignorés :", skipped)
display(manifest.groupby("doc_type").agg(fichiers=("source_path", "size"), caracteres=("n_chars", "sum")))
manifest[manifest.project_id.isna()]      # contrôle : sans n° de projet (normalement le CV)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CFG["chunking"]["size"],
    chunk_overlap=CFG["chunking"]["overlap"],
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for row in manifest.itertuples():
    text = txt_path(row.source_path).read_text(encoding="utf-8")
    if not text:
        continue
    pieces = [text] if row.no_split else splitter.split_text(text)
    for i, piece in enumerate(pieces):
        chunks.append({
            "chunk_id":    f"{row.source_path}#{i:03d}",      # chemin complet -> unique
            "text":        f"{Path(row.source_path).stem} :\n{piece}",
            "source_path": row.source_path,
            "doc_type":    row.doc_type,
            "label":       row.label,
            "project_id":  None if pd.isna(row.project_id) else int(row.project_id),
        })

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

df_chunks = pd.DataFrame(chunks)
print(f"{len(df_chunks)} chunks -> {CHUNKS_FILE.relative_to(ROOT)}")
df_chunks.groupby("doc_type").agg(n=("chunk_id", "size"),
                                  taille_moy=("text", lambda s: int(s.str.len().mean())))

## 4. Vectorisation → `index_dir` (reconstruction complète depuis `chunks_file`)

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings

with open(CHUNKS_FILE, encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

docs = [Document(page_content=r["text"], metadata={k: v for k, v in r.items() if k != "text"})
        for r in records]

embeddings = MistralAIEmbeddings(model=CFG["models"]["embedding"])
vs = FAISS.from_documents(docs, embeddings)
vs.save_local(str(INDEX_DIR))

print(f"{vs.index.ntotal} vecteurs (dim {vs.index.d}) -> {INDEX_DIR.relative_to(ROOT)}")

## 5. Test rapide (index rechargé, même `retrieve` que l'app)

In [ ]:
from src import rag

vs_test = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)

def search(question: str, k: int = CFG["retrieval"]["top_k"], **filters):
    return pd.DataFrame([
        {"distance": round(s, 3), "origin": o, "doc_type": d.metadata["doc_type"],
         "project_id": d.metadata["project_id"], "chunk_id": d.metadata["chunk_id"],
         "extrait": d.page_content[:120].replace("\n", " ")}
        for d, s, o in rag.retrieve(vs_test, question, k, filters or None)
    ])

search("Quelles technologies as-tu utilisées pour déployer une API ?")

In [ ]:
search("Qu'a pensé l'évaluateur de ce projet ?", project_id=12, doc_type="fiche")